# Vaardigheden week 8: versiebeheer en werk delen

Dit notebook hoort bij het vaardighedenspoor. Het volgt een verzonnen team van twee analisten, Ana en Ben, die reizigers tellen op een verzonnen buslijn.

Je voert echte git-opdrachten uit in een oefenmap die het notebook aanmaakt en aan het eind weer opruimt. Alles wat je ziet, is wat git ook op een laptop afdrukt.

**Zo werk je ermee:** voer elke cel op volgorde uit. Lees elke `git status`, want die vertelt je de volgende opdracht. Als je het notebook opnieuw start, voer het dan weer vanaf het begin uit.

## 1. Een oefenmap en een hulpfunctie

Git is een programma dat los van Python draait. De volgende cel maakt een tijdelijke oefenmap en een kleine functie, `git()`, die één git-opdracht in een map uitvoert en het antwoord van git afdrukt, zoals een terminal dat zou doen.

De functie laat ook de git-instellingen van je eigen computer buiten beschouwing, zodat dit notebook zich bij elke run hetzelfde gedraagt.

In [ ]:
import os, shutil, subprocess, sys, tempfile
from pathlib import Path

import numpy as np
import pandas as pd

workspace = tempfile.TemporaryDirectory(prefix="git-practice-")
root = Path(workspace.name)          # wordt in de laatste sectie weer opgeruimd
ana = root / "bus-counts"            # de projectmap van Ana
ana.mkdir()

# laat de git-instellingen van deze computer weg, zodat elke run hetzelfde verloopt
GIT_ENV = dict(os.environ, GIT_CONFIG_GLOBAL=os.devnull, GIT_CONFIG_NOSYSTEM="1")

def git(*args, where=ana, show=True):
    '''Voer één git-opdracht uit in een map, druk het antwoord van git af en geef het terug als tekst.'''
    result = subprocess.run(["git", *args], cwd=where, env=GIT_ENV, capture_output=True, text=True)
    answer = (result.stdout + result.stderr).strip()
    if show:
        print("$ git", " ".join(args))
        if answer:
            print(answer)
        print()
    return answer

git("--version")

## 2. Begin een repository

Een repository is een map met een geschiedenis eraan vast. `git init` start de geschiedenis, en `-b main` geeft de hoofdlijn van die geschiedenis de naam `main`.

Git ondertekent elke commit met een naam en een e-mailadres, dus de volgende cel stelt beide in voor deze map. Op je eigen laptop stel je ze één keer in voor alle projecten met `git config --global`. Gebruik voor werk op GitHub het privé *noreply*-adres dat GitHub aanbiedt onder Settings en dan Emails, zodat je eigen adres nooit in een openbare geschiedenis verschijnt.

In [ ]:
git("init", "-b", "main")
git("config", "user.name", "Ana")
git("config", "user.email", "ana@example.org")
git("status")

`git status` zegt dat er nog geen commits zijn en niets om te committen. De map is leeg, en git houdt haar in de gaten.

## 3. De eerste commit

Het eerste bestand van Ana bevat een telling van reizigers voor elke dag in twaalf weken. Op sommige dagen stond de teller uit, dus de telling is leeg. Op een paar dagen was de teller kapot en registreerde hij nul.

In [ ]:
rng = np.random.default_rng(8)
days = pd.date_range("2026-03-02", periods=84, freq="D")
passengers = rng.poisson(380, size=84).astype(float)
passengers[[9, 30, 31, 58]] = np.nan      # de teller stond uit
passengers[[44, 45, 70]] = 0              # de teller was kapot en registreerde nul
pd.DataFrame({"date": days.date, "passengers": passengers}).to_csv(ana / "passengers.csv", index=False)

git("status")

Het bestand is *untracked*: het staat op de schijf, maar git kent het nog niet. De tekst van git noemt zelf de volgende opdracht: `git add`.

Toevoegen zet de huidige toestand van het bestand op de lijst voor de volgende commit. Die lijst heet de *staging area*.

In [ ]:
git("add", "passengers.csv")
git("status")

Het bestand staat nu onder *Changes to be committed*, en de geschiedenis is nog steeds leeg. `git commit` maakt van alles wat klaarstaat één momentopname met een bericht.

In [ ]:
git("commit", "-m", "Voeg de dagtellingen van twaalf weken toe")
git("log", "--oneline")

Het logboek heeft één regel. De zeven tekens aan het begin zijn de korte vorm van de naam van de commit. Git berekent die naam uit de bestanden, het bericht, de auteur, het tijdstip en de commit ervoor, dus een wijziging in een van die dingen geeft een andere naam.

## 4. Code en het resultaat ervan

Ana schrijft een kort script, `analysis.py`, dat het gemiddelde aantal reizigers per dag berekent en in `results.txt` zet. De volgende cel schrijft het script in de map, voert het uit zoals een terminal dat zou doen, en commit het script samen met het resultaat.

In [ ]:
ANALYSIS = '''import pandas as pd

counts = pd.read_csv("passengers.csv")
used = counts.fillna(0)
average = used["passengers"].mean()

with open("results.txt", "w") as f:
    print("Gebruikte dagen:", len(used), file=f)
    print(f"Gemiddeld aantal reizigers per dag: {average:.1f}", file=f)
'''
(ana / "analysis.py").write_text(ANALYSIS)
subprocess.run([sys.executable, "analysis.py"], cwd=ana, check=True)
print((ana / "results.txt").read_text())

git("add", "analysis.py", "results.txt")
git("commit", "-m", "Bereken het gemiddelde aantal reizigers per dag")

## 5. Een README die een buitenstaander kan volgen

GitHub toont een bestand met de naam `README.md` op de voorpagina van elke repository, dus elke bezoeker leest het als eerste. De `.md` staat voor Markdown, een manier om platte tekst te schrijven die opgemaakte tekst wordt. Een regel die begint met `#` is een kop, een regel die begint met `-` is een opsommingspunt, en tekst tussen backticks verschijnt als code.

Een README die een buitenstaander kan volgen, beantwoordt vijf vragen: wat is dit, waar komt de data vandaan, hoe voer ik het uit, wat zit er in de map, en hoe staat het ervoor?

In [ ]:
README = '''# Reizigerstellingen van een buslijn

Dagelijkse tellingen van reizigers op één buslijn gedurende twaalf weken, en het gemiddelde per dag.

## De data
`passengers.csv` heeft één rij per dag. Een lege telling betekent dat de teller uitstond.

## Zo voer je het uit
Voer `python analysis.py` uit. Het leest `passengers.csv` en schrijft `results.txt`.

## Wat er in de map zit
- `passengers.csv`: de dagtellingen
- `analysis.py`: de analyse
- `results.txt`: het gemiddelde dat de analyse opleverde

## Status
Status: eerste analyse klaar.
'''
(ana / "README.md").write_text(README)
git("add", "README.md")
git("commit", "-m", "Schrijf een README voor een nieuwe lezer")
git("log", "--oneline")

## 6. Wijzig de analyse en lees de diff

Ana ziet dat `fillna(0)` de dagen waarop de teller uitstond, telt als dagen zonder reizigers. Ze wijzigt één regel en voert het script opnieuw uit. Voordat ze commit, laat `git diff` precies zien wat er veranderd is.

In [ ]:
ANALYSIS = ANALYSIS.replace("used = counts.fillna(0)", 'used = counts.dropna(subset=["passengers"])')
(ana / "analysis.py").write_text(ANALYSIS)
subprocess.run([sys.executable, "analysis.py"], cwd=ana, check=True)
git("status")
git("diff")

`git diff` drukte één blok af voor elk gewijzigd bestand. Een regel die begint met `-` is verwijderd, en een regel die begint met `+` is toegevoegd, dus een bewerkte regel verschijnt als één van elk.

De regel tussen de `@@`-tekens is de *hunk-header*. Neem `@@ -12,4 +12,5 @@` als voorbeeld. Het blok begint in beide bestanden op regel 12. In het oude bestand is het vier regels lang en in het nieuwe vijf.

De commit hieronder heeft een onderwerpregel en een toelichting. De tweede `-m` voegt de toelichting toe, die zegt waarom.

In [ ]:
V2_SUBJECT = "Sla dagen zonder telling over in het gemiddelde"
git("add", "analysis.py", "results.txt")
git("commit", "-m", V2_SUBJECT,
    "-m", "Een lege telling betekent dat de teller uitstond, niet dat niemand met de bus reed.")
git("log", "--oneline")

## 7. Commitberichten die vastleggen waarom

De geschiedenis is zo nuttig als haar berichten, en over zes maanden ben jij de buitenstaander die ze leest. Vier regels maken een onderwerpregel nuttig.

- Hij heeft 50 tekens of minder, zodat hij op één regel van het logboek past.
- Hij is geschreven als opdracht, in de gebiedende wijs. Schrijf *Sla dagen zonder telling over*, en vermijd *Dagen overgeslagen*.
- Hij noemt de wijziging in woorden die een lezer begrijpt. *Update README.md* en *dingen gefikst* zeggen de lezer niets.
- Als de reden niet vanzelf spreekt, zegt een toelichting na de onderwerpregel waarom.

De functie hieronder controleert de eerste drie regels. Of het bericht klopt, controleer je zelf.

In [ ]:
VAGUE = {"update", "updates", "updated", "fix", "fixes", "fixed", "change", "changes", "stuff",
         "wip", "misc", "final", "version", "readme", "readme.md", "the", "and", "of",
         "wijzigingen", "aanpassingen", "dingen", "gefikst", "definitieve", "versie", "de", "het", "en"}
PAST = {"added", "fixed", "updated", "changed", "removed", "deleted", "wrote", "made",
        "toegevoegd", "aangepast", "verwijderd", "gewijzigd", "opgelost", "gemaakt", "geschreven"}

def check_subject(subject):
    '''Geef de regels die een onderwerpregel breekt. Een lege lijst betekent dat hij slaagt.'''
    words = subject.split()
    problems = []
    if len(subject) > 50:
        problems.append("langer dan 50 tekens")
    if not subject[:1].isupper():
        problems.append("begint niet met een hoofdletter")
    if subject.endswith("."):
        problems.append("eindigt met een punt")
    if words and (words[0].lower() in PAST or words[-1].lower().strip(".") in PAST):
        problems.append("verleden tijd in plaats van een opdracht")
    if len(words) < 3 or all(w.lower().strip(".,!") in VAGUE for w in words):
        problems.append("te vaag om te zeggen wat er veranderde")
    return problems

for subject in ["Update README.md", "Dingen gefikst", "gemiddelde aangepast.", V2_SUBJECT,
                "Sla lege dagen over in het gemiddelde en geef het uitvoerbestand een duidelijkere naam"]:
    print(repr(subject), "->", check_subject(subject) or "slaagt")

### Jouw beurt 1

De lijst `bad` bevat vijf zwakke onderwerpregels uit een verzonnen geschiedenis. Het commentaar naast elke regel zegt wat die commit echt veranderde. Schrijf voor elke regel een betere onderwerpregel in de lijst `better`, in dezelfde volgorde.

Vraag je AI-assistent gerust om voorstellen, en test elk voorstel daarna met `check_subject`. Assistenten schrijven vaak *Bijgewerkt* of sluiten een onderwerpregel af met een punt.

In [ ]:
bad = [
    "Update README.md",     # de sectie toegevoegd die uitlegt hoe je de analyse uitvoert
    "dingen gefikst",       # pandas leest de datumkolom nu als datums in plaats van als tekst
    "Wijzigingen",          # een rij verwijderd die voor 14 maart twee keer voorkwam
    "Grafiek toegevoegd.",  # een grafiek toegevoegd van het gemiddelde aantal reizigers per week
    "definitieve versie",   # de resultaten voor het vervoerbureau geschreven
]

In [ ]:
# jouw beurt 1: schrijf vijf betere onderwerpregels, in dezelfde volgorde als bad
better = [
    "",
    "",
    "",
    "",
    "",
]

In [ ]:
# CHECK
assert len(better) == len(bad), "schrijf voor elk van de vijf een onderwerpregel"
for old, new in zip(bad, better):
    assert new.strip() and new != old, f"herschrijf {old!r}"
    assert check_subject(new) == [], f"{new!r}: {check_subject(new)}"
print("CHECK geslaagd: alle", len(better), "onderwerpregels volgen de regels")

## 8. Welke versie leverde dit getal op?

Een resultaat verandert telkens wanneer de analyse verandert. Als een rapport een getal noemt, vraagt iemand welke versie van de analyse het opleverde. De geschiedenis kan dat beantwoorden, omdat elk resultaat samen met de code die het maakte is gecommit.

De volgende cel schrijft een kort rapport dat het huidige gemiddelde noemt, en commit het. Daarna verandert de analyse nog één keer: het depot laat Ana weten dat de teller kapot was op de dagen waarop hij nul registreerde.

In [ ]:
average_line = (ana / "results.txt").read_text().splitlines()[-1]
(ana / "report.md").write_text("# Rapport voor het vervoerbureau\n\n" + average_line + "\n")
git("add", "report.md")
git("commit", "-m", "Schrijf het eerste rapport voor het vervoerbureau")

V3_SUBJECT = "Laat dagen weg waarop de teller nul gaf"
ANALYSIS = ANALYSIS.replace('used = counts.dropna(subset=["passengers"])',
                            'used = counts.dropna(subset=["passengers"])\nused = used[used["passengers"] > 0]')
(ana / "analysis.py").write_text(ANALYSIS)
subprocess.run([sys.executable, "analysis.py"], cwd=ana, check=True)
git("add", "analysis.py", "results.txt")
git("commit", "-m", V3_SUBJECT,
    "-m", "Volgens het depot was de teller op die dagen kapot, dus een nul is daar geen telling.")

print("report.md zegt:  ", (ana / "report.md").read_text().splitlines()[-1])
print("results.txt zegt:", (ana / "results.txt").read_text().splitlines()[-1])

Het rapport en de resultaten zijn het nu oneens, en geen van beide bestanden zegt waarom. De geschiedenis wel.

`git log -- results.txt` toont alleen de commits die `results.txt` veranderden. `git show <commit>:results.txt` drukt het bestand af zoals het in een van die commits was. De cel hieronder doet dat voor de oudste.

In [ ]:
git("log", "--oneline", "--", "results.txt")
oldest = git("log", "--format=%h", "--", "results.txt", show=False).split()[-1]
git("show", f"{oldest}:results.txt")

### Jouw beurt 2

Zoek de commit waarvan `results.txt` de regel bevat die `report.md` noemt. Sla de korte naam van de commit op in `found` en de onderwerpregel in `found_subject`.

Roep `git(..., show=False)` aan om het antwoord van git als tekst te krijgen zonder het af te drukken, en loop over de commits die `results.txt` veranderden.

In [ ]:
# jouw beurt 2: zoek de commit die het getal in report.md opleverde
found = None
found_subject = None

In [ ]:
# CHECK
assert found is not None, "found is nog leeg"
assert found_subject == V2_SUBJECT, f"{found_subject!r} leverde het getal in report.md niet op"
print("CHECK geslaagd: het rapport noemt het resultaat van", repr(found_subject))

Een tag geeft één commit een naam, zodat de volgende lezer niet hoeft te zoeken. De cel hieronder geeft de commit die het rapport opleverde een tag.

In [ ]:
git("tag", "report-1", found)
git("log", "--oneline", "--decorate")

## 9. Probeer een idee uit op een branch

Een branch is een naam die naar een commit wijst, en `main` is de standaardnaam. Als je op een nieuwe branch commit, schuift de naam van die branch mee en blijft `main` waar hij was. Een half af idee bereikt dan nooit de versie waarop je team vertrouwt. `git switch -c` maakt een branch en schakelt ernaar over.

In [ ]:
git("switch", "-c", "weekly-averages")
counts = pd.read_csv(ana / "passengers.csv", parse_dates=["date"])
counts = counts[counts["passengers"] > 0]          # dezelfde dagen die analysis.py gebruikt
weekly = counts.resample("W", on="date")["passengers"].mean().round(1)
weekly.to_csv(ana / "weekly.csv")
git("add", "weekly.csv")
git("commit", "-m", "Voeg het gemiddelde aantal reizigers per week toe")
print("bestanden op weekly-averages:", sorted(p.name for p in ana.iterdir() if not p.name.startswith(".")))
print()
git("switch", "main")
print("bestanden op main:", sorted(p.name for p in ana.iterdir() if not p.name.startswith(".")))

`weekly.csv` verdween uit de map toen Ana terugschakelde. Git heeft het bestand niet gewist: `main` wijst naar een commit die het bestand nooit had.

`git merge` haalt de commits van de branch naar `main`. Omdat `main` zelf geen nieuwe commits had, schuift git alleen de naam vooruit. Dat heet een *fast-forward*. Na de merge heeft de naam van de branch zijn werk gedaan en kan hij weg.

In [ ]:
git("merge", "weekly-averages")
git("branch", "-d", "weekly-averages")
git("log", "--oneline", "--graph")

## 10. Deel het werk met een teamgenoot

In een team staat de gedeelde kopie van een repository meestal op GitHub. Git noemt die kopie een *remote*, en de gebruikelijke naam ervoor is `origin`. Hier speelt een tweede map op deze computer de rol van GitHub, zodat je beide kanten tegelijk ziet.

`git push` stuurt je nieuwe commits naar de remote. Daarna kloont Ben de repository, waarmee hij de hele geschiedenis naar zijn eigen map kopieert. Allebei vertellen ze git ook dat het moet mergen wanneer het pullt.

In [ ]:
shared = root / "shared.git"          # speelt de rol van de repository op GitHub
git("init", "--bare", "-b", "main", str(shared), where=root)
git("remote", "add", "origin", str(shared))
git("config", "pull.rebase", "false")             # bij een pull: mergen
git("push", "-u", "origin", "main", "--tags")

ben = root / "ben-copy"
git("clone", str(shared), str(ben), where=root)
git("config", "user.name", "Ben", where=ben)
git("config", "user.email", "ben@example.org", where=ben)
git("config", "pull.rebase", "false", where=ben)
git("log", "--oneline", where=ben)

### Twee mensen veranderen één regel

Ana en Ben passen allebei de statusregel van de README aan. Ben pusht als eerste.

In [ ]:
def set_status(folder, text):
    '''Vervang de statusregel van de README in de map van één persoon.'''
    path = folder / "README.md"
    lines = ["Status: " + text if line.startswith("Status:") else line
             for line in path.read_text().splitlines()]
    path.write_text("\n".join(lines) + "\n")

set_status(ben, "wacht op de lijst van het depot met dagen waarop de teller kapot was.")
git("add", "README.md", where=ben)
git("commit", "-m", "Noteer dat we op de lijst van het depot wachten", where=ben)
git("push", where=ben)

set_status(ana, "rapport naar het vervoerbureau gestuurd.")
git("add", "README.md")
git("commit", "-m", "Leg vast dat het rapport is verstuurd")
git("push")

Git weigerde de push van Ana. Lees de hint: de remote bevat werk dat Ana niet heeft, en git stelt voor dat ze het eerst binnenhaalt met `git pull`.

Een pull haalt de commit van Ben op en voegt hem samen met die van Ana. Ze veranderden allebei dezelfde regel, dus git kan niet weten welke tekst klopt. Git voegt al het andere samen, stopt bij die regel en schrijft beide versies in het bestand.

In [ ]:
git("pull")
print((ana / "README.md").read_text())

De regels tussen `<<<<<<< HEAD` en `=======` zijn de kant van Ana, en de regels tussen `=======` en `>>>>>>>` zijn die van Ben.

Een conflict oplossen betekent: beide kanten lezen, schrijven wat de regel moet zeggen, de drie markeringsregels wissen en committen. Hier kloppen beide uitspraken, dus de opgeloste regel bewaart beide feiten.

In [ ]:
text = (ana / "README.md").read_text()
start = text.index("<<<<<<<")
end = text.index("\n", text.index(">>>>>>>")) + 1
resolved = "Status: rapport verstuurd; wacht op de lijst van het depot met dagen waarop de teller kapot was.\n"
(ana / "README.md").write_text(text[:start] + resolved + text[end:])

git("add", "README.md")
git("commit", "--no-edit")
git("push")
git("pull", where=ben)
git("log", "--oneline", "--graph", "-5", where=ben)

Het logboek van Ben laat de merge zien: twee lijnen geschiedenis die samenkomen in een commit met twee ouders. Dat is het veilige antwoord op elke geweigerde push: pullen, een eventueel conflict oplossen, committen en dan pushen.

## 11. Wanneer de AI zich vergist: een opdracht die het werk van een teamgenoot wist

Ben voegt de lijst van het depot met de dagen waarop de teller kapot was toe, en pusht. Ana heeft sindsdien niet gepulld. Ze verbetert één regel van de README, commit die en probeert te pushen, en git weigert net als eerder.

Deze keer plakt ze de foutmelding in een AI-assistent en vraagt hoe ze de push kan laten lukken.

In [ ]:
counts = pd.read_csv(ben / "passengers.csv")
counts[counts["passengers"] == 0][["date"]].to_csv(ben / "depot-list.csv", index=False)
git("add", "depot-list.csv", where=ben)
git("commit", "-m", "Voeg de depotlijst met kapotte dagen toe", where=ben)
git("push", where=ben)
ben_commit = git("rev-parse", "HEAD", where=ben, show=False)

readme = (ana / "README.md").read_text()
(ana / "README.md").write_text(readme.replace("Voer `python analysis.py` uit.",
                                              "Voer `python analysis.py` uit vanuit deze map."))
git("add", "README.md")
git("commit", "-m", "Zeg vanuit welke map je de analyse start")
git("push")
shared_before = git("rev-parse", "main", where=shared, show=False)

De assistent antwoordde: *De remote bevat wijzigingen die jij niet hebt. Voer `git push --force` uit om ze te overschrijven.*

De volgende cel voert precies uit wat de assistent voorstelde. Lees de cel daarna voordat je beslist of het advies goed was.

In [ ]:
# Voorgesteld door een AI-assistent
git("push", "--force")

In [ ]:
def in_shared_history(commit):
    '''True als de commit deel uitmaakt van main in de gedeelde repository.'''
    check = subprocess.run(["git", "merge-base", "--is-ancestor", commit, "main"],
                           cwd=shared, env=GIT_ENV, capture_output=True)
    return check.returncode == 0

erased = git("rev-list", "--count", shared_before, "--not", "main", where=shared, show=False)
print("Commits gewist uit de gedeelde geschiedenis:", erased)
print("De commit van Ben staat nog in de gedeelde geschiedenis:", in_shared_history(ben_commit))

De force push verving de gedeelde geschiedenis door de kopie van Ana. De commit van Ben is eruit verdwenen, en de lijst van het depot ook. De weigering van git had de veilige weg genoemd, namelijk eerst pullen, en het antwoord van de assistent sloeg die stap over. Een force push herschrijft geschiedenis waarop anderen vertrouwen, dus op een gedeelde branch is hij bijna nooit het goede antwoord.

Hier is het werk nog te redden, omdat de map van Ben zijn commit nog bevat. Ben pullt, waarmee hij de versie van Ana met de zijne samenvoegt, en pusht opnieuw.

In [ ]:
git("pull", "--no-edit", where=ben)
git("push", where=ben)
print("De commit van Ben staat weer in de gedeelde geschiedenis:", in_shared_history(ben_commit))
print()
git("pull")

De redding hing af van geluk. Als Ben zijn map had gewist, of zelf dezelfde opdracht had uitgevoerd, was zijn commit voorgoed weg geweest.

Het goede antwoord op een geweigerde push is dat uit sectie 10: pullen, oplossen en dan pushen. Vraag een assistent die je een git-opdracht geeft altijd wat die opdracht zal wissen, voordat je hem uitvoert.

## 12. Wanneer de AI zich vergist: een commit die een wachtwoord publiceert

De map van Ana bevat nu nog twee bestanden die nooit gecommit mogen worden. `.env` bewaart het wachtwoord van de database met tellingen, en `raw_counter_export.csv` is een grote ruwe export van de tellers.

Ze vraagt een AI-assistent: *Hoe sla ik al mijn werk op en deel ik het met het team?*

In [ ]:
(ana / ".env").write_text("COUNTER_DB_PASSWORD=verzonnen-wachtwoord-om-te-oefenen\n")
pd.DataFrame({"reading": rng.integers(0, 1_000_000, size=300_000)}).to_csv(
    ana / "raw_counter_export.csv", index=False)

def size_mb(path):
    '''Grootte van een bestand, of van alles in een map, in megabytes.'''
    files = [path] if path.is_file() else [p for p in path.rglob("*") if p.is_file()]
    return round(sum(p.stat().st_size for p in files) / 1e6, 1)

print("Grootte van de ruwe export in MB:", size_mb(ana / "raw_counter_export.csv"))
print("Grootte van de opgeslagen geschiedenis vooraf, in MB:", size_mb(ana / ".git" / "objects"))
print()
git("status")

De volgende cel voert het antwoord van de assistent precies zo uit als het geschreven is.

In [ ]:
# Voorgesteld door een AI-assistent
git("add", ".")
git("commit", "-m", "update")
git("push")

`git add .` zette alles in de map klaar wat git niet moest negeren. Kijk wat de commit bevat, en daarna naar de map van Ben nadat hij heeft gepulld.

In [ ]:
git("show", "--stat", "--format=%s", "HEAD")
git("pull", where=ben)
print("De kopie van .env bij Ben:", (ben / ".env").read_text().strip())

Ana probeert het ongedaan te maken. Ze laat git beide bestanden niet langer volgen, schrijft een `.gitignore` waarin ze staan, commit en pusht. `git rm --cached` haalt een bestand uit de volgende commit en laat het op de schijf staan.

In [ ]:
git("rm", "--cached", ".env", "raw_counter_export.csv")
(ana / ".gitignore").write_text(".env\nraw_counter_export.csv\n")
git("add", ".gitignore")
git("commit", "-m", "Volg het wachtwoordbestand en de ruwe export niet")
git("push")
git("ls-files")

De twee bestanden zijn uit de nieuwste commit verdwenen. De commit daarvoor bevat ze nog steeds.

In [ ]:
leaked = git("show", "HEAD~1:.env", show=False)
print("Wachtwoord nog leesbaar in de geschiedenis:", "PASSWORD" in leaked)
print(leaked)
print("Grootte van de opgeslagen geschiedenis na het verwijderen, in MB:", size_mb(ana / ".git" / "objects"))

Een latere commit haalt een bestand nooit uit de commits ervoor. Ben heeft het wachtwoord nu, en bij een openbare repository heeft iedereen die keek het misschien ook. De enige veilige reactie op een gepusht geheim is het als gelekt te behandelen en meteen te veranderen. Het uit de geschiedenis halen vraagt speciale hulpmiddelen en de hulp van elke teamgenoot, en kopieën die al gemaakt zijn, bereik je er niet mee. Ook de ruwe export blijft in de geschiedenis, dus elke toekomstige kloon downloadt hem.

Twee gewoonten voorkomen beide problemen. Schrijf de `.gitignore` vóór de eerste `git add`, en lees `git status` vóór elke commit.

### Jouw beurt 3: een schone start

De volgende cel maakt een nieuwe map, `clean-start`, met dezelfde bestanden, waaronder het wachtwoordbestand en de ruwe export, en start daarin een repository.

Maak een eerste commit die alleen de bestanden bevat die erin horen. Schrijf eerst een `.gitignore`, zet de juiste bestanden klaar, lees `git status` en commit met een onderwerpregel die `check_subject` doorstaat. Geef `where=clean` mee aan elke aanroep van `git()`.

In [ ]:
clean = root / "clean-start"
clean.mkdir()
for name in ["README.md", "analysis.py", "passengers.csv", "results.txt", ".env", "raw_counter_export.csv"]:
    shutil.copy(ana / name, clean / name)
git("init", "-b", "main", where=clean)
git("config", "user.name", "Ana", where=clean)
git("config", "user.email", "ana@example.org", where=clean)
git("status", where=clean)

In [ ]:
# jouw beurt 3: schrijf clean / ".gitignore", zet de juiste bestanden klaar en commit, steeds met where=clean

In [ ]:
# CHECK
tracked = set(git("ls-tree", "-r", "--name-only", "HEAD", where=clean, show=False).split())
assert ".env" not in tracked, "het wachtwoordbestand zit in de commit"
assert "raw_counter_export.csv" not in tracked, "de ruwe export zit in de commit"
assert tracked == {".gitignore", "README.md", "analysis.py", "passengers.csv", "results.txt"}, tracked
subject = git("log", "-1", "--format=%s", where=clean, show=False)
assert check_subject(subject) == [], f"{subject!r}: {check_subject(subject)}"
assert "!! .env" in git("status", "--ignored", "--porcelain", where=clean, show=False)
print("CHECK geslaagd: de eerste commit bevat", len(tracked), "bestanden, en git negeert het wachtwoordbestand")

## 13. Dezelfde opdrachten op GitHub

Alles hierboven werkt op dezelfde manier met een repository op GitHub. Er veranderen drie dingen.

- De remote is een webadres zoals `https://github.com/jouw-team/jouw-repository.git`, en `git clone` neemt dat adres.
- Bij een push vraagt GitHub om een *personal access token* in plaats van je wachtwoord. Maak een fine-grained token voor één repository met een korte geldigheid, en typ hem alleen in een verborgen invoer zoals `getpass` van Python. Een token die je in een cel of in het adres van de remote typt, wordt opgeslagen op een plek waar anderen hem kunnen vinden.
- Iedereen kan een openbare repository lezen, dus daar tellen de `.gitignore` en de gewoonte van `git status` nog zwaarder.

## 14. Opruimen

De laatste cel ruimt de oefenmap op, met alles erin.

In [ ]:
workspace.cleanup()
print("Oefenmap opgeruimd:", not root.exists())

## Wat je hebt gedaan

Je hebt een repository gestart, bestanden klaargezet en gecommit, een diff gelezen en berichten geschreven die zeggen waarom. Je vond de versie die een getal opleverde en gaf die een tag, je mergede een branch, je deelde werk via een remote en je loste een conflict op.

Je zag ook twee voorstellen van een AI schade aanrichten. Een force push wiste de commit van een teamgenoot, en een `git add .` publiceerde een wachtwoord. Het meeste andere in git is een variatie op de opdrachten in dit notebook.